## Init Environment

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

## Feature Engineering

### 1.Day Past Due 

In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def dpd_dim_ften(df: DataFrame) -> DataFrame:
    """
    Engineer delinquency severity and behavior features.

    Preserves the original columns and returns a new DataFrame
    with additional candidate features.
    """
    d30 = "NumberOfTime30-59DaysPastDueNotWorse"
    d60 = "NumberOfTime60-89DaysPastDueNotWorse"
    d90 = "NumberOfTimes90DaysLate"

    # Cast source columns to numeric types
    df = (
        df
        .withColumn(d30, F.col(d30).cast("double"))
        .withColumn(d60, F.col(d60).cast("double"))
        .withColumn(d90, F.col(d90).cast("double"))
    )

    # 1. Total delinquency burden
    df = df.withColumn(
        "delinq_total",
        F.col(d30) + F.col(d60) + F.col(d90)
    )

    # 2. Any delinquency
    df = df.withColumn(
        "delinq_any_flag",
        F.when(F.col("delinq_total").isNull(), None)
         .when(F.col("delinq_total") > 0, 1)
         .otherwise(0)
    )

    # 3. Any delinquency at 60+ days
    df = df.withColumn(
        "delinq_60plus_flag",
        F.when(F.col(d60).isNull() | F.col(d90).isNull(), None)
         .when((F.col(d60) > 0) | (F.col(d90) > 0), 1)
         .otherwise(0)
    )

    # 4. Any delinquency at 90+ days
    df = df.withColumn(
        "delinq_90plus_flag",
        F.when(F.col(d90).isNull(), None)
         .when(F.col(d90) > 0, 1)
         .otherwise(0)
    )

    # 5. Weighted severity score (candidate weights: 1, 2, 3)
    df = df.withColumn(
        "delinq_severity",
        F.col(d30) + 2 * F.col(d60) + 3 * F.col(d90)
    )

    # 6. Severe delinquency share
    severe_count = F.coalesce(F.col(d60), F.lit(0.0)) + \
                   F.coalesce(F.col(d90), F.lit(0.0))

    df = df.withColumn(
        "delinq_severe_share",
        F.when(
            F.col("delinq_total").isNull(),
            None
        ).otherwise(
            severe_count / (F.col("delinq_total") + 1)
        )
    )

    # 7. Repeated delinquency indicator
    df = df.withColumn(
        "delinq_repeated_flag",
        F.when(F.col("delinq_total").isNull(), None)
         .when(F.col("delinq_total") >= 2, 1)
         .otherwise(0)
    )

    return df

### 2. UR 

In [3]:
def engineer_utilization_features(df: DataFrame) -> DataFrame:
    """
    Dimension 2: Revolving Credit Pressure & Utilization.

    Creates candidate features while preserving the original columns.
    """
    util = "RevolvingUtilizationOfUnsecuredLines"

    # Ensure numeric type
    df = df.withColumn(
        util,
        F.col(util).cast("double")
    )

    x = F.col(util)

    # 1. Log transformation: compress large positive values
    df = df.withColumn(
        "utilization_log1p",
        F.when(x >= 0, F.log1p(x))
    )

    # 2. High utilization indicators
    df = (
        df
        .withColumn(
            "utilization_over_80_flag",
            F.when(x.isNull(), None)
             .when(x > 0.8, 1)
             .otherwise(0)
        )
        .withColumn(
            "utilization_over_100_flag",
            F.when(x.isNull(), None)
             .when(x > 1.0, 1)
             .otherwise(0)
        )
    )

    # 3. Utilization bands
    df = df.withColumn(
        "utilization_band",
        F.when(x.isNull(), "unknown")
         .when(x < 0, "negative")
         .when(x == 0, "zero")
         .when(x <= 0.3, "low")
         .when(x <= 0.6, "moderate")
         .when(x <= 0.8, "elevated")
         .when(x <= 1.0, "high")
         .otherwise("over_limit")
    )

    # 4. Cap extreme values for a separate modeling candidate
    df = df.withColumn(
        "utilization_capped",
        F.when(x.isNull(), None)
         .otherwise(F.least(x, F.lit(1.0)))
    )

    # 5. Extreme-value indicator
    df = df.withColumn(
        "utilization_extreme_flag",
        F.when(x.isNull(), None)
         .when(x > 10.0, 1)
         .otherwise(0)
    )

    return df

### Draft Training

## Draf Test Zone

In [23]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 439.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 81.2 kB/s eta 0:00:0000:0100:42
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 218.7 kB/s eta 0:00:0000:0100:41
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 183.7 kB/s eta 0:00:0000:0100:05

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [7]:
%cd /home/jovyan/work/
from spark.spark_utils import CustomSparkSession
spark_conf = CustomSparkSession(
    app_name="Noah"
)
spark = spark_conf.create_session()

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 14:25:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [23]:
df = spark.read\
.option("header",True)\
.option("inferSchema",True)\
.option("nullValue","NA")\
.csv(
    "s3a://sandbox/GiveMeSomeCredit/cs-training.csv/"
)\
.withColumnRenamed(
    "_c0","id"
)

In [27]:
df = dpd_dim_ften(df)
df = engineer_utilization_features(df)
df = (
    df
    .withColumn("MonthlyIncome", F.col("MonthlyIncome").cast("double"))        # "NA" -> null
    .withColumn("NumberOfDependents", F.col("NumberOfDependents").cast("double"))
    .withColumn("utilization_band",F.col("utilization_band").cast("double"))
)

In [28]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

TARGET = "SeriousDlqin2yrs"
df_pd = df.toPandas()
feature_columns= [c for c in df.columns if c not in [TARGET, "id"]]
X = df_pd[feature_columns]
y = df_pd[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # drop if balanced
    eval_metric="auc",
    early_stopping_rounds=50,
    tree_method="hist",
    enable_categorical=True,  # needs category dtype for cat cols
    n_jobs=-1,
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=50,
)

pred = model.predict_proba(X_valid)[:, 1]
print(f"Valid AUC: {roc_auc_score(y_valid, pred):.4f} | best iter: {model.best_iteration}")

[0]	validation_0-auc:0.86001	validation_1-auc:0.85233
[50]	validation_0-auc:0.87781	validation_1-auc:0.86786
[100]	validation_0-auc:0.88725	validation_1-auc:0.86891
[144]	validation_0-auc:0.89414	validation_1-auc:0.86794
Valid AUC: 0.8689 | best iter: 94


In [26]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- SeriousDlqin2yrs: integer (nullable = true)
 |-- RevolvingUtilizationOfUnsecuredLines: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- NumberOfTime30-59DaysPastDueNotWorse: double (nullable = true)
 |-- DebtRatio: double (nullable = true)
 |-- MonthlyIncome: double (nullable = true)
 |-- NumberOfOpenCreditLinesAndLoans: integer (nullable = true)
 |-- NumberOfTimes90DaysLate: double (nullable = true)
 |-- NumberRealEstateLoansOrLines: integer (nullable = true)
 |-- NumberOfTime60-89DaysPastDueNotWorse: double (nullable = true)
 |-- NumberOfDependents: double (nullable = true)
 |-- delinq_total: double (nullable = true)
 |-- delinq_any_flag: integer (nullable = true)
 |-- delinq_60plus_flag: integer (nullable = true)
 |-- delinq_90plus_flag: integer (nullable = true)
 |-- delinq_severity: double (nullable = true)
 |-- delinq_severe_share: double (nullable = true)
 |-- delinq_repeated_flag: integer (nullable = true)
 |-- utili

## Test Dataset

In [30]:
test_df = spark.read\
.option("header",True)\
.option("inferSchema",True)\
.option("nullValue","NA")\
.csv(
    "s3a://sandbox/GiveMeSomeCredit/cs-test.csv/"
)\
.withColumnRenamed(
    "_c0","id"
)
test_df = dpd_dim_ften(test_df)
test_df = engineer_utilization_features(test_df)
test_df = (
    test_df
    .withColumn("MonthlyIncome", F.col("MonthlyIncome").cast("double"))        # "NA" -> null
    .withColumn("NumberOfDependents", F.col("NumberOfDependents").cast("double"))
    .withColumn("utilization_band",F.col("utilization_band").cast("double"))
)

## Result Plotting

In [34]:
import pandas as pd

train_cats = pd.Categorical(X_train["utilization_band"]).categories
print(train_cats)   # sanity: should list your band labels

test_pd = test_df.toPandas()
test_pd["utilization_band"] = pd.Categorical(test_pd["utilization_band"], categories=train_cats)

X_kaggle = test_pd[feature_columns]
print(X_kaggle["utilization_band"].dtype)   # category
print(test_pd["utilization_band"].isna().sum())   # unseen/null bands

test_pd["Probability"] = model.predict_proba(X_kaggle)[:, 1]
test_pd[["id", "Probability"]].rename(columns={"id": "Id"}).to_csv("submission.csv", index=False)

Index([], dtype='float64')
category
101503


XGBoostError: [14:41:57] /__w/xgboost/xgboost/src/data/columnar.h:134: Category index from DataFrame has floating point dtype, consider using strings or integers instead.
Stack trace:
  [bt] (0) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(+0x2c1a8c) [0x760a7e6c1a8c]
  [bt] (1) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(+0x4e3a08) [0x760a7e8e3a08]
  [bt] (2) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(+0x4e534d) [0x760a7e8e534d]
  [bt] (3) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(+0x4e5478) [0x760a7e8e5478]
  [bt] (4) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(+0x4dd494) [0x760a7e8dd494]
  [bt] (5) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(+0x6897b5) [0x760a7ea897b5]
  [bt] (6) /usr/local/lib/python3.11/site-packages/xgboost/lib/libxgboost.so(XGBoosterPredictFromColumnar+0xbf) [0x760a7e5d1ddf]
  [bt] (7) /lib/x86_64-linux-gnu/libffi.so.8(+0x6f7a) [0x